# Notebook 10 — Editorial revision evidence

## Reliability-qualified daily nighttime lights: temporal scale, observability, and configuration selection

This notebook is the evidence package for the methodological revision. It is organized around two reviewer concerns:

1. **Does the 30-day centred window conceal the behaviour of the native daily signal?**
2. **Can the configuration-selection rule be simplified, justified, and transferred across sub-grids?**

The analysis therefore begins with **native-daily observations (`W=1`)**, treats window length as a sensitivity parameter rather than a fixed default, and separates three distinct quantities:

- **Pearson `r`** — alignment with the NGCP load series;
- **`N%`** — the percentage of dates with a valid paired estimate;
- **composite score** — `N × r²`, used only to rank configurations after the two components have been reported separately.

### Recommended publication package

| Role | Evidence | Recommended location |
|---|---|---|
| Establish the native-daily baseline | Samar–Leyte `r`, direct-date `N%`, and composite-score surfaces | Main paper |
| Quantify what smoothing changes | Fixed-configuration window-effect summary | Main paper |
| Show the practical signal consequence | Haiyan and Rai alignment at selected windows | Main paper |
| Demonstrate the framework across sub-grids | Regional `r–N%` trade-off and selected configurations | Main paper |
| Summarize transferability | Compact regional score surface and best configuration per sub-grid | Main paper or supplementary if space is constrained |
| Document robustness | Checkpoint heatmaps, component surfaces, and score-form sensitivity | Supplementary |
| Preserve auditability | Dense regional surfaces and full trajectory diagnostics | Diagnostic archive only |

**Reading note.** Code is retained for reproducibility but collapsed by default. Each result is preceded by its publication role, reviewer-comment mapping, interpretation, and cautious claim.


## Reviewer-evidence map

| Reviewer concern | Evidence in this notebook | What it permits us to claim |
|---|---|---|
| **R1.1 / R2.6 — 30-day smoothing** | Native-daily baseline; fixed-setting window profile; event-window comparison; checkpoint sensitivity | Thirty days is not treated as universally optimal. Longer windows improve temporal continuity but represent temporally supported estimates rather than more direct observations. |
| **R2.2 / R3.3 — parameter sensitivity** | Five-percent spatial-completeness sweeps across GHSL masks and window checkpoints | Threshold and mask effects are visible and application-dependent; selection is a documented decision surface rather than an assumed constant. |
| **R1.2 / R3.4 — composite-score weighting** | Separate `r` and `N%` reporting; `N × r²` ranking; `N × r` versus `N × r²` sensitivity | The revised score has an explicit interpretation, contains no RMSE duplication, and its effect on configuration choice is transparent. |
| **R2.3 — regional transferability** | Five-sub-grid component, score, and trade-off results | The workflow transfers across the Visayas sub-grids, while the selected mask and threshold remain context-dependent. This is not evidence of global transferability. |
| **R3.6 / R3.7 — sparse coverage and high correlation** | `r–N%` trade-off curves and direct-date coverage | High correlation is not presented as sufficient when retained temporal coverage is low. |
| **R1.3 / R2.1 / E1 — novelty** | Ordered framework: observe → qualify → align → diagnose trade-off → select → test sensitivity | The contribution is the formalized reliability workflow and its decision evidence, not a claim that each individual metric is new. |

Reviewer IDs refer to the consolidated revision tracker. Items not resolved by these analyses are listed at the end of the notebook.


## 0. Reproducibility and data provenance

The notebook discovers the existing GHSL-mask exports and NGCP load series, validates the common daily index, and records the source files used for each sub-grid. No methodological conclusion should be interpreted before the registry checks complete successfully.

**Output location:** all publication figures, supplementary figures, diagnostic plots, and tables are written to the accessible `revision_plots/` folder beside this notebook.


In [28]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

DEFAULT_ROOT = Path(
    "/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/"
    "02 - CH1 - Reliability-Qualified Nighttime Lights (Technical Validation)/"
    "blackmarble-reliability-conditions"
)
ROOT = Path(os.environ.get("RQ1_PROJECT_ROOT", DEFAULT_ROOT)).expanduser()
OUTPUT_ROOT = Path(
    os.environ.get(
        "RQ1_NOTEBOOK10_OUTPUT",
        Path("/Users/S4135723/Documents/ChatGPT/Milestone 1/revision_plots"),
    )
).expanduser()
FIG_MAIN = OUTPUT_ROOT / "01_main_manuscript"
FIG_SUPP = OUTPUT_ROOT / "02_supplementary"
FIG_DIAGNOSTIC = OUTPUT_ROOT / "03_diagnostic_archive"
TABLE_MAIN = OUTPUT_ROOT / "04_tables" / "main_manuscript"
TABLE_SUPP = OUTPUT_ROOT / "04_tables" / "supplementary"
for folder in (FIG_MAIN, FIG_SUPP, FIG_DIAGNOSTIC, TABLE_MAIN, TABLE_SUPP):
    folder.mkdir(parents=True, exist_ok=True)

# Match Notebook 04's multi-subgrid analysis period and input locations.
DATE_START = pd.Timestamp("2012-01-19")
DATE_END = pd.Timestamp("2024-12-31")
VALIDATION_DATES = pd.date_range(DATE_START, DATE_END, freq="D", name="date")
NTL_FOLDER = ROOT / "datasets" / "vnp46a2" / "visayas_ghsl_masked_daily_stats"
NGCP_XLSX_PATH = ROOT / "datasets" / "ngcp" / "NGCP_hourly_load.xlsx"

WINDOWS = np.arange(1, 31)
WINDOW_CHECKPOINTS = [1, 4, 7, 14, 21, 30]
SC_GRID = np.arange(0, 101, 1, dtype=float)  # exact search and CSV output
MAIN_TAUS = np.arange(0, 101, 5, dtype=float)  # retain the paper's display interval
SUPP_TAUS = MAIN_TAUS.copy()                   # consistent figures throughout
PRIMARY_REGION = "Samar-Leyte"
EVENTS = [
    ("Haiyan", pd.Timestamp("2013-11-08"), "2013-05-01", "2014-05-31"),
    ("Rai", pd.Timestamp("2021-12-16"), "2021-07-01", "2022-07-31"),
]

GROUPS = [
    ("G8", "10-30", "All Land", "#4E79A7"),
    ("G7", "11-30", "Rural Settlements", "#A8CE74"),
    ("G6", "12-30", "Low–Mid Rural", "#7BAF54"),
    ("G5", "13-30", "Built-Up / Rural Mix", "#375623"),
    ("G4", "21-30", "Urban Fringe", "#D9B900"),
    ("G3", "22-30", "Core & Inner", "#E6A23C"),
    ("G2", "23-30", "Dense Urban", "#A87000"),
    ("G1", "30", "Urban Core", "#D65A4A"),
]
GROUP_LOOKUP = {
    code: {"suffix": suffix, "label": label, "color": color}
    for code, suffix, label, color in GROUPS
}
ORDERED_MASKS = [f"G{i}" for i in range(1, 9)]
MASK_COLORS = {code: GROUP_LOOKUP[code]["color"] for code in ORDERED_MASKS}

REGION_RENAME = {
    "CebuProvince": "Cebu",
    "SamarLeyte": "Samar-Leyte",
    "NegrosIsland": "Negros",
    "PanayIsland": "Panay",
    "BoholProvince": "Bohol",
}
REGION_ORDER = ["Samar-Leyte", "Bohol", "Cebu", "Negros", "Panay"]
GROUP_RENAME = {"30-only": "30"}
SHEET_MAP = {
    "Panay": "PANAY HOURLY LOAD 2013-2024",
    "Negros": "NEGROS HOURLY LOAD 2013-2024",
    "Samar-Leyte": "LEY-SAM HOURLY LOAD 2013-2024",
    "Cebu": "CEBU HOURLY LOAD 2013-2024",
    "Bohol": "BOHOL LOAD 2013-2024",
}

PLOT_TEMPLATE = "plotly_white"
INK = "#233B60"
GRID = "#E8EDF3"
R_CONTRAST_SCALE = "jet_r"
N_CONTRAST_SCALE = "Viridis"
SCORE_CONTRAST_SCALE = "Turbo"
DIVERGING_SCALE = [[0.00, "#B2182B"], [0.50, "#F7F7F7"], [1.00, "#2166AC"]]
POSITIVE_SCALE = "Viridis"
R_COLOUR_POWER = 2.0
N_COLOUR_POWER = 3.0
R_TICKS = np.array([-1.0, -0.5, 0.0, 0.5, 0.75, 1.0])
R_COLOUR_TICKS = np.sign(R_TICKS) * np.abs(R_TICKS) ** R_COLOUR_POWER
N_TICKS = np.array([0, 50, 75, 90, 100], dtype=float)
N_COLOUR_TICKS = (N_TICKS / 100) ** N_COLOUR_POWER

def save_figure(fig, folder, stem, width=None, height=None):
    fig.write_html(folder / f"{stem}.html", include_plotlyjs="cdn")
    try:
        kwargs = {"scale": 2}
        if width is not None:
            kwargs["width"] = width
        if height is not None:
            kwargs["height"] = height
        fig.write_image(str(folder / f"{stem}.png"), **kwargs)
    except Exception as exc:
        print(f"{stem}: HTML saved; PNG unavailable ({exc})")
    return fig


In [29]:
def parse_filename(filepath):
    name = Path(filepath).name.replace("_stats.csv", "")
    parts = name.split("_DNBBRDF_class")
    if len(parts) != 2:
        return "Unknown", "Unknown"
    region, group = parts
    return REGION_RENAME.get(region, region), GROUP_RENAME.get(group, group)


def read_multiregion_ntl(folder):
    csv_files = sorted(Path(folder).glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No NTL CSV files found in {folder}")

    frames = []
    for path in csv_files:
        region, group = parse_filename(path)
        if region == "Unknown":
            continue
        try:
            frame = pd.read_csv(path)
        except Exception:
            frame = pd.read_csv(path, dtype=str, engine="python", on_bad_lines="skip")

        if "date" not in frame.columns and "system:index" in frame.columns:
            frame["date"] = (
                frame["system:index"].astype(str)
                .str.extract(r"(\d{4}_\d{2}_\d{2})")[0]
                .str.replace("_", "-", regex=False)
            )
        if "date" not in frame.columns:
            raise ValueError(f"{path.name}: neither date nor system:index is available")

        frame["Region"] = region
        frame["Group"] = group
        frame["source_file"] = str(path)
        frames.append(frame)

    if not frames:
        raise RuntimeError(f"No recognised multi-region NTL files were loaded from {folder}")

    panel = pd.concat(frames, ignore_index=True)
    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize()
    panel = panel[panel["date"].between(DATE_START, DATE_END)].copy()
    panel["Valid_px"] = pd.to_numeric(panel["Valid_px"], errors="coerce")
    panel["Mean_by_sum"] = pd.to_numeric(panel["Mean_by_sum"], errors="coerce")
    return panel.sort_values(["Region", "Group", "date"]).reset_index(drop=True)


def mask_frame(group_frame):
    data = group_frame.copy().drop_duplicates("date").set_index("date").sort_index()
    valid = data["Valid_px"]
    raw = data["Mean_by_sum"].where(valid > 0)
    denominator = valid.loc[raw.notna()].max()
    if not np.isfinite(denominator) or denominator <= 0:
        raise ValueError("No valid direct observations for this region–mask combination")
    p95 = raw.quantile(0.95)
    return pd.DataFrame({
        "raw": raw,
        "ntl": raw.clip(upper=p95),
        "SC": 100 * valid / denominator,
        "Valid_px": valid,
    }).reindex(VALIDATION_DATES)


def load_ngcp_grid_1am_flat(xlsx_path, sheet_map):
    series_by_region = {}
    selected_columns = {}
    for region, sheet_name in sheet_map.items():
        frame = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=1)
        frame.columns = [str(column).strip() for column in frame.columns]
        date_col = frame.columns[0]
        frame = frame.rename(columns={date_col: "DATE"})
        frame["DATE"] = pd.to_datetime(frame["DATE"], errors="coerce").dt.normalize()
        frame = frame.dropna(subset=["DATE"]).set_index("DATE")
        frame.columns = [str(column).strip() for column in frame.columns]

        possible = ["1", "01", "1.0", "Hour 1", "HOUR 1", "1AM", "1am"]
        hour_col = next((column for column in possible if column in frame.columns), None)
        if hour_col is None:
            numeric_like = [
                column for column in frame.columns
                if str(column).replace(".0", "").isdigit()
            ]
            hour_col = numeric_like[0] if numeric_like else None
        if hour_col is None and len(frame.columns):
            hour_col = frame.columns[0]
        if hour_col is None:
            raise KeyError(f"No usable 01:00 load column in sheet {sheet_name}")

        values = pd.to_numeric(frame[hour_col], errors="coerce")
        values = values.groupby(level=0).mean().reindex(VALIDATION_DATES)
        series_by_region[region] = values.rename(region)
        selected_columns[region] = str(hour_col)

    return pd.DataFrame(series_by_region), selected_columns


NTL_PANEL = read_multiregion_ntl(NTL_FOLDER)
NGCP_PANEL, NGCP_COLUMNS = load_ngcp_grid_1am_flat(NGCP_XLSX_PATH, SHEET_MAP)

DATA_REGISTRY = {}
registry_rows = []
for region in REGION_ORDER:
    masks = {}
    source_files = []
    for code in ORDERED_MASKS:
        group = GROUP_LOOKUP[code]["suffix"]
        subset = NTL_PANEL[
            NTL_PANEL["Region"].eq(region) & NTL_PANEL["Group"].eq(group)
        ].copy()
        if subset.empty:
            raise FileNotFoundError(
                f"{region}: GHSL group {group} ({code}) is absent from {NTL_FOLDER}"
            )
        masks[code] = mask_frame(subset)
        source_files.extend(subset["source_file"].dropna().unique().tolist())

    load = NGCP_PANEL[region].copy()
    DATA_REGISTRY[region] = {"masks": masks, "load": load}
    registry_rows.append({
        "region": region,
        "ntl_files": len(set(source_files)),
        "ghsl_masks": len(masks),
        "ngcp_sheet": SHEET_MAP[region],
        "load_column": NGCP_COLUMNS[region],
        "load_dates": int(load.notna().sum()),
    })

REGISTRY_TABLE = pd.DataFrame(registry_rows)
display(REGISTRY_TABLE)
REGISTRY_TABLE.to_csv(TABLE_SUPP / "data_registry.csv", index=False)


/var/folders/91/f5wz25_x3r586p828vt1hm080000gp/T/ipykernel_24749/2218596871.py:74: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  frame["DATE"] = pd.to_datetime(frame["DATE"], errors="coerce").dt.normalize()
/var/folders/91/f5wz25_x3r586p828vt1hm080000gp/T/ipykernel_24749/2218596871.py:74: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  frame["DATE"] = pd.to_datetime(frame["DATE"], errors="coerce").dt.normalize()
/var/folders/91/f5wz25_x3r586p828vt1hm080000gp/T/ipykernel_24749/2218596871.py:74: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  frame["DATE"] = pd.to_datetime(frame[

,region,ntl_files,ghsl_masks,ngcp_sheet,load_column,load_dates
0,Samar-Leyte,8,8,LEY-SAM HOURLY LOAD 2013-2024,Hour No.,4383
1,Bohol,8,8,BOHOL LOAD 2013-2024,Hour No.,4383
2,Cebu,8,8,CEBU HOURLY LOAD 2013-2024,Hour No.,4383
3,Negros,8,8,NEGROS HOURLY LOAD 2013-2024,Hour No.,4383
4,Panay,8,8,PANAY HOURLY LOAD 2013-2024,1,4383


# 1. Temporal aggregation and the 30-day-window concern

## Analytical question

How do spatial completeness, GHSL mask, and centred window length change alignment (`r`) and usable paired coverage (`N%`)?

## Revision decision

- Use **`W=1` as the primary configuration-selection scale** so that reliability is assessed on the native daily signal.
- Treat `W=4, 7, 14, 21, 30` as **sensitivity checkpoints**, not interchangeable daily observations.
- Retain `W=30` only when the estimand is explicitly a smoothed, retrospective representation.

This directly addresses **R1.1, R2.2, R2.6, R3.3, R3.5, R3.6, and R3.7**.


In [16]:
def paired_metrics(ntl, load, direct_mask, window_days):
    frame = pd.DataFrame({"ntl": ntl, "load": load}).dropna()
    n_possible = int(load.notna().sum())
    n_pairs = len(frame)
    r = np.nan
    if (
        n_pairs >= 3
        and frame.ntl.std(ddof=0) > 0
        and frame.load.std(ddof=0) > 0
    ):
        r = float(frame.ntl.corr(frame.load))
    direct_n = int((direct_mask & load.notna()).sum())
    return {
        "n_pairs": n_pairs,
        "N_pct": 100 * n_pairs / n_possible if n_possible else np.nan,
        "direct_n": direct_n,
        "direct_pct": 100 * direct_n / n_possible if n_possible else np.nan,
        "r": r,
        "score": (n_pairs / n_possible) * max(r, 0) ** 2 if n_possible and np.isfinite(r) else 0.0,
    }

def evaluate_region(region, windows=WINDOWS, sc_grid=SC_GRID):
    load = DATA_REGISTRY[region]["load"]
    rows = []
    for code in ORDERED_MASKS:
        data = DATA_REGISTRY[region]["masks"][code]
        for tau in sc_grid:
            direct_mask = data.ntl.notna() & data.SC.ge(tau)
            qualified = data.ntl.where(direct_mask)
            direct_counts = direct_mask.astype(int)
            for width in windows:
                ntl_mean = qualified.rolling(int(width), min_periods=1, center=True).mean()
                load_mean = load.rolling(int(width), min_periods=1, center=True).mean()
                count = direct_counts.rolling(int(width), min_periods=1, center=True).sum()
                metrics = paired_metrics(ntl_mean, load_mean, direct_mask, int(width))
                paired = pd.DataFrame({"ntl": ntl_mean, "load": load_mean, "count": count}).dropna()
                rows.append({
                    "region": region,
                    "mask": code,
                    "mask_label": GROUP_LOOKUP[code]["label"],
                    "tau": float(tau),
                    "window_days": int(width),
                    "mean_direct_days": float(paired["count"].mean()) if len(paired) else np.nan,
                    "mean_window_support_pct": (
                        100 * float(paired["count"].mean()) / width if len(paired) else np.nan
                    ),
                    **metrics,
                })
    return pd.DataFrame(rows)

SENS_PRIMARY = evaluate_region(PRIMARY_REGION)
SENS_PRIMARY.to_csv(TABLE_SUPP / "samar_leyte_mask_sc_window_sensitivity.csv.gz", index=False, compression="gzip")
print(f"{len(SENS_PRIMARY):,} primary-region configurations calculated")


24,240 primary-region configurations calculated


## 1.1 Main-paper evidence — native-daily reliability surface

**Reviewer IDs:** R1.1, R1.2, R2.2, R2.6, R3.3, R3.4, R3.6, R3.7; supports R1.3, R2.1, and E1.

**Placement:** Main paper.

**How to read the result.** Each cell is one GHSL mask–spatial-completeness configuration at `W=1`. The first panel shows alignment, the second shows the percentage of dates actually retained, and the third shows their combined ranking. Reading the panels together prevents a configuration with high `r` but sparse coverage from being presented as automatically reliable.

**Main-paper interpretation.** The native-daily surface reveals a structured trade-off rather than one universally optimal threshold. Urban-focused masks generally preserve stronger alignment, whereas stricter spatial-completeness requirements reduce the number of usable dates. The composite score identifies the balance within this observed decision surface; it does not replace the component metrics.

**Cautious claim.** These results justify the framework's sequence—diagnose observability first, then evaluate alignment and select a configuration—not a universal acceptance threshold.


In [38]:
# Display grid retained from Notebook 3: τ = 5, 10, ..., 95%
DISPLAY_TAUS_5 = np.arange(5, 100, 5, dtype=float)


def mask_tau_matrix(
    frame,
    metric,
    window_days=1,
    region=PRIMARY_REGION,
    display_taus=DISPLAY_TAUS_5,
):
    subset = frame[
        frame.region.eq(region)
        & frame.window_days.eq(window_days)
    ]

    return (
        subset
        .pivot(index="mask", columns="tau", values=metric)
        .reindex(index=ORDERED_MASKS, columns=display_taus)
    )


# Retained because later supplementary figures use these helpers.
def heatmap_colour_values(metric, values):
    values = np.asarray(values, dtype=float)

    if metric == "r":
        return np.sign(values) * np.abs(values) ** R_COLOUR_POWER

    if metric == "N_pct":
        return (np.clip(values, 0, 100) / 100) ** N_COLOUR_POWER

    return values


def heatmap_style(metric, score_max=None):
    if metric == "r":
        return dict(
            colorscale=R_CONTRAST_SCALE,
            cmin=-1,
            cmax=1,
            ticks=R_COLOUR_TICKS,
            ticktext=[f"{value:g}" for value in R_TICKS],
            title="Pearson r",
        )

    if metric == "N_pct":
        return dict(
            colorscale=N_CONTRAST_SCALE,
            cmin=0,
            cmax=1,
            ticks=N_COLOUR_TICKS,
            ticktext=[f"{value:g}%" for value in N_TICKS],
            title="N%",
        )

    maximum = max(0.01, float(score_max or 1))

    return dict(
        colorscale=SCORE_CONTRAST_SCALE,
        cmin=0,
        cmax=maximum,
        ticks=None,
        ticktext=None,
        title="Score",
    )


def plot_daily_metric_heatmap(frame, metric):
    """
    Reproduces the style of Notebook 3, Figure 6.

    Panels:
        r       = Pearson correlation
        N_pct   = retained direct-date coverage
        score   = (N% / 100) × max(r, 0)^2
    """

    matrix = mask_tau_matrix(
        frame,
        metric,
        window_days=1,
        region=PRIMARY_REGION,
        display_taus=DISPLAY_TAUS_5,
    )

    z = matrix.to_numpy(dtype=float)

    y_labels = [
        f"{code}: {GROUP_LOOKUP[code]['label']}"
        for code in matrix.index
    ]

    if metric == "r":
        colorscale = "Blues_r"
        zmin, zmax = 0, 0.60
        highlight = z >= 0.60
        text_format = "{:.2f}"

        colorbar = dict(
            title=dict(text="Pearson r", side="right"),
            len=0.85,
            tickvals=[0, 0.20, 0.40, 0.60],
            ticktext=["0", "0.20", "0.40", "≥0.60"],
        )

        stem = "section1_native_daily_pearson_r"

    elif metric == "N_pct":
        colorscale = "Greens_r"
        zmin, zmax = 0, 50
        highlight = z >= 50
        text_format = "{:.0f}"

        colorbar = dict(
            title=dict(text="N (%)", side="right"),
            len=0.85,
            tickvals=[0, 10, 20, 30, 40, 50],
            ticktext=["0", "10", "20", "30", "40", "≥50"],
        )

        stem = "section1_native_daily_direct_N"

    elif metric == "score":
        colorscale = "Pinkyl_r"
        zmin = 0
        zmax = max(0.01, float(np.nanmax(z)))

        finite_scores = z[np.isfinite(z)]
        top_decile = (
            np.nanpercentile(finite_scores, 90)
            if finite_scores.size
            else np.nan
        )

        highlight = z >= top_decile
        text_format = "{:.2f}"

        score_ticks = np.linspace(zmin, zmax, 5)

        colorbar = dict(
            title=dict(text="Composite score", side="right"),
            len=0.85,
            tickvals=score_ticks,
            ticktext=[f"{value:.2f}" for value in score_ticks],
        )

        stem = "section1_native_daily_composite_score"

    else:
        raise ValueError(
            "metric must be 'r', 'N_pct', or 'score'"
        )

    fig = go.Figure()

    fig.add_trace(
        go.Heatmap(
            z=z,
            x=matrix.columns,
            y=y_labels,
            colorscale=colorscale,
            zmin=zmin,
            zmax=zmax,
            xgap=2,
            ygap=2,
            colorbar=colorbar,
            showscale=True,
            hovertemplate=(
                "Mask %{y}<br>"
                "Spatial completeness τ=%{x:.0f}%<br>"
                "Value=%{z:.3f}"
                "<extra></extra>"
            ),
        )
    )

    # Annotate every cell; bold favourable configurations.
    for row in range(z.shape[0]):
        for col in range(z.shape[1]):
            value = z[row, col]

            if not np.isfinite(value):
                continue

            value_text = text_format.format(value)

            if highlight[row, col]:
                value_text = f"<b>{value_text}</b>"

            fig.add_annotation(
                x=matrix.columns[col],
                y=y_labels[row],
                text=value_text,
                showarrow=False,
                font=dict(
                    family="Arial",
                    size=14,
                    color="black",
                ),
            )

    fig.update_layout(
        template="plotly_white",
        width=1500,
        height=400,
        margin=dict(
            l=175,
            r=45,
            t=20,
            b=80,
        ),
        xaxis=dict(
            title="Spatial completeness threshold, τ (%)",
            tickvals=list(range(0, 101, 20)),
            tickfont=dict(size=14),
        ),
        yaxis=dict(
            tickfont=dict(size=14),
        ),
        font=dict(
            family="Arial",
            color=INK,
            size=16,
        ),
        paper_bgcolor="rgba(0,0,0,0)",
    )

    return fig, stem


# ------------------------------------------------------------------
# Main Figure panels: r, direct-date N%, and the revised composite
# ------------------------------------------------------------------

fig_10a, stem_10a = plot_daily_metric_heatmap(
    SENS_PRIMARY,
    "r",
)

fig_10b, stem_10b = plot_daily_metric_heatmap(
    SENS_PRIMARY,
    "N_pct",
)

fig_10c, stem_10c = plot_daily_metric_heatmap(
    SENS_PRIMARY,
    "score",
)

fig_10a.show()
fig_10b.show()
fig_10c.show()

MAIN_DAILY_R_PANEL = save_figure(
    fig_10a,
    FIG_MAIN,
    stem_10a,
    1500,
    400,
)

MAIN_DAILY_N_PANEL = save_figure(
    fig_10b,
    FIG_MAIN,
    stem_10b,
    1500,
    400,
)

MAIN_DAILY_SCORE_PANEL = save_figure(
    fig_10c,
    FIG_MAIN,
    stem_10c,
    1500,
    400,
)


# ------------------------------------------------------------------
# Highest-scoring Samar–Leyte configurations on the 5% τ grid
# ------------------------------------------------------------------

DAILY_COMPONENT_ALL = (
    SENS_PRIMARY[
        SENS_PRIMARY.window_days.eq(1)
    ][
        [
            "mask",
            "mask_label",
            "tau",
            "r",
            "n_pairs",
            "N_pct",
            "direct_pct",
            "score",
        ]
    ]
    .copy()
)

DAILY_COMPONENT_5PCT = (
    DAILY_COMPONENT_ALL[
        DAILY_COMPONENT_ALL.tau.isin(DISPLAY_TAUS_5)
    ]
    .sort_values(
        ["score", "N_pct", "r", "tau", "mask"],
        ascending=[False, False, False, True, True],
    )
    .reset_index(drop=True)
)

TOP_SAMAR_LEYTE_5PCT = (
    DAILY_COMPONENT_5PCT
    .head(10)
    .copy()
)

TOP_SAMAR_LEYTE_5PCT.insert(
    0,
    "rank",
    np.arange(1, len(TOP_SAMAR_LEYTE_5PCT) + 1),
)

display(
    Markdown(
        "### Highest-scoring native-daily Samar–Leyte "
        "configurations (5% τ grid)"
    )
)

display(
    TOP_SAMAR_LEYTE_5PCT.round(
        {
            "tau": 0,
            "r": 3,
            "N_pct": 1,
            "direct_pct": 1,
            "score": 4,
        }
    )
)

TOP_SAMAR_LEYTE_5PCT.to_csv(
    TABLE_MAIN
    / "section1_top10_samar_leyte_daily_5pct_tau.csv",
    index=False,
)

DAILY_COMPONENT_5PCT.to_csv(
    TABLE_MAIN
    / "section1_samar_leyte_daily_5pct_tau_all.csv",
    index=False,
)

# Retain the full 1% results for audit and sensitivity reporting.
DAILY_COMPONENT_ALL.to_csv(
    TABLE_SUPP
    / "section1_samar_leyte_daily_exact_1pct_tau_all.csv",
    index=False,
)

### Highest-scoring native-daily Samar–Leyte configurations (5% τ grid)

,rank,mask,mask_label,tau,r,n_pairs,N_pct,direct_pct,score
0,1,G1,Urban Core,25.0,0.602,1936,44.2,44.2,0.1600
1,2,G1,Urban Core,30.0,0.615,1823,41.6,41.6,0.1575
2,3,G1,Urban Core,20.0,0.579,2057,46.9,46.9,0.1572
3,4,G1,Urban Core,15.0,0.559,2191,50.0,50.0,0.1560
4,5,G1,Urban Core,40.0,0.650,1605,36.6,36.6,0.1546
5,6,G1,Urban Core,10.0,0.537,2334,53.3,53.3,0.1537
6,7,G1,Urban Core,35.0,0.624,1698,38.7,38.7,0.1508
7,8,G2,Dense Urban,15.0,0.522,2340,53.4,53.4,0.1453
8,9,G2,Dense Urban,25.0,0.561,2016,46.0,46.0,0.1448
9,10,G1,Urban Core,45.0,0.652,1489,34.0,34.0,0.1445


## 1.2 Main-paper evidence — what changes when the window increases?

**Reviewer IDs:** R1.1, R2.2, R2.6, R3.5, R3.6.

**Placement:** Main paper.

The daily-optimal mask and spatial-completeness threshold are fixed before varying `W`. This isolates the temporal-window effect from configuration changes.

**How to read the result.** Direct-observation `N%` is determined by valid observations on the target date. Paired-estimate `N%` can rise with `W` because a date becomes estimable when at least one supported observation falls inside its centred window. Thus, a value near 100% at a long window does **not** mean that every day was directly observed, nor that only one value exists per 30 days. The rolling estimate is still calculated for each calendar date, but neighbouring dates reuse overlapping observations.

**Main-paper interpretation.** Increasing the window creates a smoother and more continuous representation while progressively weakening day-specific meaning. The gain is therefore an estimand change—greater temporal support in exchange for temporal precision—not the recovery of missing daily observations.


In [63]:
DAILY_PREVIEW = SENS_PRIMARY[SENS_PRIMARY.window_days.eq(1)].sort_values(
    ["score", "N_pct", "tau", "mask"], ascending=[False, False, True, True]
)
REFERENCE = DAILY_PREVIEW.iloc[0]
REFERENCE_MASK = REFERENCE["mask"]
REFERENCE_TAU = float(REFERENCE["tau"])
WINDOW_PROFILE = SENS_PRIMARY[
    SENS_PRIMARY["mask"].eq(REFERENCE_MASK)
    & SENS_PRIMARY["tau"].eq(REFERENCE_TAU)
].sort_values("window_days")
WINDOW_CHECKPOINT_TABLE = WINDOW_PROFILE[
    WINDOW_PROFILE.window_days.isin(WINDOW_CHECKPOINTS)
][[
    "window_days", "mask", "tau", "r", "n_pairs", "N_pct",
    "direct_pct", "mean_direct_days", "mean_window_support_pct",
]]
display(WINDOW_CHECKPOINT_TABLE.round(3))
WINDOW_CHECKPOINT_TABLE.to_csv(TABLE_MAIN / "section1_window_checkpoint_table.csv", index=False)

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=.12,
    subplot_titles=["Pearson r", "Paired-estimate N% and direct-observation N%"],
)
fig.add_trace(go.Scatter(
    x=WINDOW_PROFILE.window_days, y=WINDOW_PROFILE.r, mode="lines+markers",
    name="Pearson r", line=dict(color="#2952CC", width=2.8),
    hovertemplate="W=%{x} d<br>r=%{y:.3f}<extra></extra>",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=WINDOW_PROFILE.window_days, y=WINDOW_PROFILE.N_pct, mode="lines+markers",
    name="Paired-estimate N%", line=dict(color="#D65A4A", width=2.8),
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=WINDOW_PROFILE.window_days, y=WINDOW_PROFILE.direct_pct, mode="lines",
    name="Direct-observation N%", line=dict(color=INK, width=2.2, dash="dash"),
), row=2, col=1)
for row, metric, color in [(1, "r", "#2952CC"), (2, "N_pct", "#D65A4A")]:
    check = WINDOW_PROFILE[WINDOW_PROFILE.window_days.isin(WINDOW_CHECKPOINTS)]
    fig.add_trace(go.Scatter(
        x=check.window_days, y=check[metric], mode="markers+text",
        text=[f"W={int(w)}" if w in (1, 4, 30) else "" for w in check.window_days],
        textposition="top center", showlegend=False,
        marker=dict(symbol="circle-open", size=10, color=color, line=dict(width=2)),
        hoverinfo="skip",
    ), row=row, col=1)
fig.update_xaxes(title_text="Centered common calendar window (days)", tickvals=WINDOW_CHECKPOINTS, row=2, col=1)
fig.update_yaxes(title_text="Pearson r", range=[-1, 1.03], gridcolor=GRID, row=1, col=1)
fig.update_yaxes(title_text="N%", range=[0, 102], gridcolor=GRID, row=2, col=1)
fig.update_layout(
    template=PLOT_TEMPLATE, width=1250, height=760,
    title=f"Window effect after fixing {REFERENCE_MASK} · SC τ={REFERENCE_TAU:.0f}% from W=1",
    legend=dict(orientation="h", y=-.12), margin=dict(l=80, r=40, t=90, b=95),
    font=dict(family="Arial", color=INK, size=17),
    paper_bgcolor="white", plot_bgcolor="white",
)
fig.show()
MAIN_WINDOW_EFFECT = save_figure(fig, FIG_MAIN, "section1_window_effect_summary", 1250, 760)


,window_days,mask,tau,r,n_pairs,N_pct,direct_pct,mean_direct_days,mean_window_support_pct
750,1,G1,25.0,0.602,1936,44.171,44.171,1.000,100.000
753,4,G1,25.0,0.697,3342,76.232,44.183,2.319,57.974
756,7,G1,25.0,0.734,3837,87.483,44.209,3.537,50.531
763,14,G1,25.0,0.786,4222,96.195,44.224,6.434,45.958
770,21,G1,25.0,0.817,4349,98.998,44.207,9.378,44.659
779,30,G1,25.0,0.837,4384,99.704,44.189,13.312,44.373


## 1.3 Main-paper evidence — event-period signal representation

**Reviewer IDs:** R1.1 and R2.6; supports discussion of R2.4, R2.5, and R3.5 without constituting independent event validation.

**Placement:** Main paper for `W=1, 4, 30`; supplementary for the full checkpoint set.

**How to read the result.** The storm date is a temporal reference, not a causal breakpoint estimate. `W=1` retains the observed daily gaps and volatility. Short windows improve legibility while preserving relatively local timing. `W=30` produces the most continuous curve, but the centred window distributes information across days before and after the event.

**Main-paper interpretation.** The figure makes the reviewer's concern visible: smoothing can aid comparison with NGCP load, yet it also broadens and attenuates rapid change. Window choice must therefore follow the intended application. Retrospective monthly-scale assessment may tolerate longer windows; event-timing or near-real-time use requires the native or short-window signal and explicit missingness.

**Do not claim:** that these panels estimate outage magnitude, recovery duration, or causal storm effects.


In [70]:
# ============================================================
# Event-period NTL–NGCP alignment
# Main: W = 1, 4, 30
# Supplement: W = 1, 4, 7, 14, 30
# ============================================================

TAUS_5 = np.arange(5, 100, 5)

EVENT_CASES = [
    dict(event="Haiyan", region="Samar-Leyte", date="2013-11-08",
         left="2013-05-01", right="2014-05-31"),
    dict(event="Rai", region="Bohol", date="2021-12-16",
         left="2021-07-01", right="2022-07-31"),
]


def best_daily_config(region):
    q = DAILY_ALL[
        DAILY_ALL.region.eq(region)
        & DAILY_ALL.window_days.eq(1)
        & DAILY_ALL.tau.isin(TAUS_5)
    ]
    if q.empty:
        raise ValueError(f"No daily configuration found for {region}")

    return q.sort_values(
        ["score", "N_pct", "r", "tau", "mask"],
        ascending=[False, False, False, True, True],
    ).iloc[0]


BEST_EVENT_CONFIGS = {
    case["region"]: best_daily_config(case["region"])
    for case in EVENT_CASES
}


def paired_zscores(ntl, load):
    paired = pd.DataFrame({"ntl": ntl, "load": load}).dropna()

    if len(paired) < 3 or paired.ntl.std(ddof=0) == 0 or paired.load.std(ddof=0) == 0:
        return (
            pd.Series(np.nan, index=ntl.index),
            pd.Series(np.nan, index=load.index),
        )

    return (
        (ntl - paired.ntl.mean()) / paired.ntl.std(ddof=0),
        (load - paired.load.mean()) / paired.load.std(ddof=0),
    )


def event_signal_figure(windows, folder, stem, width):
    ncols = len(windows)
    annotation_size = 13 if ncols == 5 else 14

    fig = make_subplots(
        rows=2, cols=ncols, shared_yaxes=True,
        horizontal_spacing=.025 if ncols == 5 else .05,
        vertical_spacing=.19,
    )

    for row, case in enumerate(EVENT_CASES, start=1):
        region, event = case["region"], case["event"]
        event_date = pd.Timestamp(case["date"])
        dates = pd.date_range(case["left"], case["right"], freq="D")

        best = BEST_EVENT_CONFIGS[region]
        mask, tau = best["mask"], float(best["tau"])
        mask_data = DATA_REGISTRY[region]["masks"][mask]
        load = DATA_REGISTRY[region]["load"]

        direct_mask = mask_data.ntl.notna() & mask_data.SC.ge(tau)
        qualified_ntl = mask_data.ntl.where(direct_mask)

        for col, w in enumerate(windows, start=1):
            ntl_w = qualified_ntl.rolling(w, min_periods=1, center=True).mean()
            load_w = load.rolling(w, min_periods=1, center=True).mean()
            support_w = qualified_ntl.notna().astype(int).rolling(
                w, min_periods=1, center=True
            ).sum()

            ntl_z, load_z = paired_zscores(ntl_w, load_w)
            event_pairs = pd.DataFrame({
                "ntl": ntl_z.reindex(dates),
                "load": load_z.reindex(dates),
            }).dropna()

            event_r = (
                event_pairs.ntl.corr(event_pairs.load)
                if len(event_pairs) >= 3 else np.nan
            )

            load_available = load.reindex(dates).notna()
            possible_n = int(load_available.sum())
            paired_pct = 100 * len(event_pairs) / possible_n if possible_n else np.nan

            direct_available = (
                direct_mask.reindex(dates).fillna(False).astype(bool)
                & load_available
            )
            direct_pct = (
                100 * int(direct_available.sum()) / possible_n
                if possible_n else np.nan
            )

            support = support_w.reindex(event_pairs.index)
            mean_support = float(support.mean()) if len(support) else np.nan
            r_text = f"{event_r:.2f}" if np.isfinite(event_r) else "—"

            # SC-qualified NTL
            fig.add_trace(
                go.Scatter(
                    x=dates, y=ntl_z.reindex(dates), mode="lines",
                    name="SC-qualified NTL", legendgroup="ntl",
                    showlegend=(row == 1 and col == 1),
                    line=dict(color="#2952CC", width=2.6),
                    connectgaps=False,
                    customdata=support_w.reindex(dates),
                    hovertemplate=(
                        "%{x|%d %b %Y}<br>NTL z = %{y:.2f}"
                        "<br>Direct observations in window = %{customdata:.0f}"
                        "<extra>SC-qualified NTL</extra>"
                    ),
                ),
                row=row, col=col,
            )

            # NGCP load
            fig.add_trace(
                go.Scatter(
                    x=dates, y=load_z.reindex(dates), mode="lines",
                    name="NGCP load at 01:00", legendgroup="load",
                    showlegend=(row == 1 and col == 1),
                    line=dict(color="#202020", width=2.3),
                    connectgaps=False,
                    hovertemplate=(
                        "%{x|%d %b %Y}<br>NGCP z = %{y:.2f}"
                        "<extra>NGCP load at 01:00</extra>"
                    ),
                ),
                row=row, col=col,
            )

            fig.add_vline(
                x=event_date, row=row, col=col,
                line_dash="dash", line_color=INK, line_width=1.5,
            )

            axis_n = (row - 1) * ncols + col
            xref = "x domain" if axis_n == 1 else f"x{axis_n} domain"
            yref = "y domain" if axis_n == 1 else f"y{axis_n} domain"

            # Window is the annotation heading; metrics are secondary.
            fig.add_annotation(
                x=.985, y=.97, xref=xref, yref=yref,
                text=(
                    f"W = {w} days"
                    f"<br><i>r</i><sub>event</sub> = {r_text}"
                    f"<br><i>N</i><sub>paired</sub> = {paired_pct:.1f}%"
                    f" · <i>N</i><sub>direct</sub> = {direct_pct:.1f}%"
                    f"<br>mean support = {mean_support:.1f}/{w} d"
                ),
                showarrow=False, xanchor="right", yanchor="top", align="right",
                bgcolor="rgba(255,255,255,.88)",
                bordercolor="#A9B9C9", borderwidth=.8, borderpad=5,
                font=dict(family="Arial", size=annotation_size, color=INK),
            )

            fig.update_xaxes(
                tickformat="%b\n%Y", nticks=6,
                tickfont=dict(family="Arial", size=13),
                gridcolor=GRID, showline=True, linecolor=INK,
                linewidth=1.2, ticks="outside", ticklen=4,
                row=row, col=col,
            )

        # One centred scientific title per event row.
        fig.add_annotation(
            x=.5, y=1.07 if row == 1 else .50,
            xref="paper", yref="paper",
            text=f"{event} · {region} · {mask} · τ = {tau:.0f}%",
            showarrow=False, xanchor="center", yanchor="middle",
            font=dict(family="Arial", size=21, color=INK),
        )

    fig.update_yaxes(
        range=[-4, 4], tickvals=[-4, -2, 0, 2, 4],
        tickfont=dict(family="Arial", size=14),
        gridcolor=GRID, zeroline=True, zerolinecolor="#A9B9C9",
        showline=True, linecolor=INK, linewidth=1.2,
        ticks="outside", ticklen=4,
    )

    for row in (1, 2):
        fig.update_yaxes(
            title_text="Standardised signal (z)",
            title_font=dict(family="Arial", size=17, color=INK),
            row=row, col=1,
        )

    fig.update_layout(
        template="plotly_white",
        width=width, height=800,
        title=None,

        # Horizontal legend aligned with the first row title.
        legend=dict(
            orientation="h",
            x=.99, xanchor="right",
            y=1.07, yanchor="middle",
            font=dict(family="Arial", size=15, color=INK),
            bgcolor="rgba(255,255,255,.8)",
        ),

        margin=dict(l=100, r=35, t=90, b=65),
        font=dict(family="Arial", color=INK, size=15),
        paper_bgcolor="white", plot_bgcolor="white",
    )

    fig.show()
    return save_figure(fig, folder, stem, width, 800)


# Compact main-manuscript version
MAIN_EVENT_SIGNAL = event_signal_figure(
    windows=[1, 4, 30],
    folder=FIG_MAIN,
    stem="section1_event_alignment_main_W1_W4_W30",
    width=1700,
)

# Complete supplementary sensitivity version
SUPP_EVENT_SIGNAL = event_signal_figure(
    windows=[1, 4, 7, 14, 21, 30],
    folder=FIG_SUPP,
    stem="section1_event_alignment_supp_W1_W4_W7_W14_W30",
    width=2400,
)

## 1.4 Supplementary evidence — checkpoint sensitivity surfaces

**Reviewer IDs:** R1.1, R1.7, R2.2, R2.6, R3.3, R3.5, R3.6.

**Placement:** Supplementary materials.

**How to read the result.** Moving from left to right changes only the centred window. Moving vertically changes the GHSL thematic mask, and moving horizontally changes the spatial-completeness threshold. The `r` panels show whether smoothing changes apparent alignment; the `N%` panels show how rapidly paired estimates become available as temporal support expands.

**Supplementary interpretation.** The broad pattern is stable: longer windows increase paired-estimate continuity, especially at moderate thresholds, but the response of `r` differs by mask and threshold. This demonstrates why the paper reports sensitivity surfaces rather than presenting 30 days as a universally defensible default.


In [71]:
# ============================================================
# Supplementary checkpoint heatmaps
# W = 1, 4, 7, 14, 21, and 30 days
# ============================================================

DISPLAY_TAUS_5 = np.arange(5, 100, 5)
CHECKPOINT_WINDOWS = [1, 4, 7, 14, 21, 30]

FULL_MASK_LABELS = {
    code: f"{code}: {GROUP_LOOKUP[code]['label']}"
    for code in ORDERED_MASKS
}


def checkpoint_heatmaps(metric):
    is_r = metric == "r"
    metric_title = "Pearson r" if is_r else "Paired-estimate N%"
    colorscale = "Jet" if is_r else "Viridis"
    cmin, cmax = (0, 1) if is_r else (0, 100)
    ticks = [0, .25, .5, .75, 1] if is_r else [0, 25, 50, 75, 100]
    ticktext = ["0", ".25", ".50", ".75", "1"] if is_r else ["0", "25", "50", "75", "100"]

    fig = make_subplots(
        rows=1, cols=6, shared_yaxes=True,
        horizontal_spacing=.025,
        subplot_titles=[f"W = {w} d" for w in CHECKPOINT_WINDOWS],
    )

    for heading in fig.layout.annotations:
        heading.font = dict(family="Arial", size=19, color=INK)

    for col, window in enumerate(CHECKPOINT_WINDOWS, start=1):
        subset = SENS_PRIMARY[SENS_PRIMARY.window_days.eq(window)]

        matrix = (
            subset.pivot(index="mask", columns="tau", values=metric)
            .reindex(index=ORDERED_MASKS, columns=DISPLAY_TAUS_5)
        )

        raw = matrix.to_numpy(dtype=float)
        colour_values = np.clip(raw, 0, 1) if is_r else raw
        value_hover = (
            "Pearson r=%{customdata:.3f}"
            if is_r else
            "Paired-estimate N=%{customdata:.1f}%"
        )

        fig.add_trace(
            go.Heatmap(
                x=matrix.columns,
                y=[FULL_MASK_LABELS[code] for code in matrix.index],
                z=colour_values,
                customdata=raw,
                coloraxis="coloraxis",
                xgap=.8,
                ygap=.8,
                hoverongaps=False,
                hovertemplate=(
                    f"W={window} days<br>"
                    "Mask=%{y}<br>"
                    "SC threshold τ=%{x:.0f}%<br>"
                    + value_hover
                    + "<extra></extra>"
                ),
            ),
            row=1, col=col,
        )

        fig.update_xaxes(
            range=[2.5, 97.5],
            title_text="SC threshold, τ (%)",
            title_font=dict(family="Arial", size=17, color=INK),
            tickvals=[20, 40, 60, 80],
            ticktext=["20", "40", "60", "80"],
            tickfont=dict(family="Arial", size=15, color=INK),
            showline=True, mirror=True,
            linecolor=INK, linewidth=1.4,
            ticks="outside", ticklen=5,
            row=1, col=col,
        )

        fig.update_yaxes(
            title_text="GHSL thematic mask" if col == 1 else None,
            title_font=dict(family="Arial", size=18, color=INK),
            tickfont=dict(family="Arial", size=14, color=INK),
            showticklabels=(col == 1),
            showline=True, mirror=True,
            linecolor=INK, linewidth=1.4,
            ticks="outside", ticklen=5,
            row=1, col=col,
        )

    # Centred metric heading.
    fig.add_annotation(
        x=.5, y=1.13,
        xref="paper", yref="paper",
        text=metric_title,
        showarrow=False,
        xanchor="center", yanchor="middle",
        font=dict(family="Arial", size=23, color=INK),
    )

    # Compact colour bar at the upper-right.
    colorbar = dict(
        orientation="h",
        x=.91, xanchor="center",
        y=1.13, yanchor="middle",
        len=.13, thickness=25,
        tickvals=ticks, ticktext=ticktext,
        tickfont=dict(family="Arial", size=14, color=INK),
        ticks="outside", ticklen=4,
        outlinecolor=INK, outlinewidth=1,
    )

    fig.update_layout(
        template="plotly_white",
        width=2300, height=570,
        title=None,
        coloraxis=dict(
            colorscale=colorscale,
            cmin=cmin, cmax=cmax,
            colorbar=colorbar,
        ),
        margin=dict(l=225, r=60, t=105, b=85),
        font=dict(family="Arial", color=INK, size=16),
        paper_bgcolor="white",
        plot_bgcolor="#E9EDF2",
    )

    fig.show()

    tag = "r" if is_r else "N_pct"
    return save_figure(
        fig, FIG_SUPP,
        f"section1_checkpoint_heatmaps_{tag}",
        2300, 570,
    )


SUPP_CHECKPOINT_R = checkpoint_heatmaps("r")
SUPP_CHECKPOINT_N = checkpoint_heatmaps("N_pct")

## 1.5 Diagnostic archive — dense surfaces and trajectory checks

**Publication role:** retain for auditability; do not include in the main paper because these plots repeat the checkpoint result with greater visual density.

- The full mask-by-window surfaces confirm the progression across every day from `W=1` to `W=30`.
- The trajectory panels provide a direct QA view of monotonic and non-monotonic responses at selected thresholds.
- If space is available, one may be cited as an online diagnostic; otherwise the checkpoint heatmaps carry the same reviewer-facing message more efficiently.


In [62]:
# ============================================================
# Supplementary sensitivity surface — compact final version
# ============================================================

TAUS_5 = np.arange(5, 100, 5)
WINDOWS_1_30 = np.arange(1, 31)

GAP = 0.17
LEFT_CENTER = (1 - GAP) / 4
RIGHT_CENTER = 1 - LEFT_CENTER

FONT = dict(family="Arial", color=INK)
AXIS_LINE = dict(showline=True, mirror=True, linecolor=INK, linewidth=1.4,
                 ticks="outside", ticklen=5, tickwidth=1.2)


def sensitivity_matrix(code, metric):
    return (
        SENS_PRIMARY[SENS_PRIMARY["mask"].eq(code)]
        .pivot(index="window_days", columns="tau", values=metric)
        .reindex(index=WINDOWS_1_30, columns=TAUS_5)
    )


def mask_progression_surface():
    fig = make_subplots(
        rows=8, cols=2, shared_xaxes=True, shared_yaxes=True,
        horizontal_spacing=GAP, vertical_spacing=0.022,
    )

    metric_specs = [
        ("r", 1, "coloraxis", "Pearson r"),
        ("N_pct", 2, "coloraxis2", "Paired-estimate N%"),
    ]

    for row, code in enumerate(ORDERED_MASKS, start=1):
        for metric, col, coloraxis, label in metric_specs:
            matrix = sensitivity_matrix(code, metric)
            raw = matrix.to_numpy(dtype=float)

            # Negative r values receive the r=0 colour only.
            colour_values = np.clip(raw, 0, 1) if metric == "r" else raw

            fig.add_trace(
                go.Heatmap(
                    x=matrix.columns, y=matrix.index,
                    z=colour_values, customdata=raw,
                    coloraxis=coloraxis, xgap=0.7, ygap=0,
                    hoverongaps=False,
                    hovertemplate=(
                        f"{code} · {GROUP_LOOKUP[code]['label']}<br>"
                        "SC τ=%{x:.0f}%<br>W=%{y:.0f} days<br>"
                        f"{label}=%{{customdata:.3f}}<extra></extra>"
                    ),
                ),
                row=row, col=col,
            )

        for col in (1, 2):
            fig.update_xaxes(
                range=[2.5, 97.5],
                tickvals=[20, 40, 60, 80],
                ticktext=["20", "40", "60", "80"],
                showticklabels=(row == 8),
                title_text="Spatial completeness threshold, τ (%)" if row == 8 else None,
                title_font={**FONT, "size": 21},
                tickfont={**FONT, "size": 16},
                tickangle=0, row=row, col=col, **AXIS_LINE,
            )

            fig.update_yaxes(
                range=[0.5, 30.5],
                tickvals=[1, 4, 7, 14, 21, 30],
                showticklabels=(col == 1),
                title_text="W (days)" if col == 1 else None,
                title_font={**FONT, "size": 17},
                tickfont={**FONT, "size": 15},
                row=row, col=col, **AXIS_LINE,
            )

        # Horizontal GHSL label centred in the gap.
        fig.add_annotation(
            x=0.5, y=1 - (row - 0.5) / 8,
            xref="paper", yref="paper",
            text=f"{code} · {GROUP_LOOKUP[code]['label']}",
            showarrow=False, xanchor="center", yanchor="middle",
            font={**FONT, "size": 17},
        )

    # Column headings centred over their respective panels.
    for x, text in [
        (LEFT_CENTER, "Pearson r"),
        (RIGHT_CENTER, "Paired-estimate N%"),
    ]:
        fig.add_annotation(
            x=x, y=1.075, xref="paper", yref="paper",
            text=text, showarrow=False,
            xanchor="center", yanchor="middle",
            font={**FONT, "size": 25},
        )

    shared_colorbar = dict(
        orientation="h", y=1.025, yanchor="middle",
        len=0.16, thickness=28,
        tickfont={**FONT, "size": 15},
        ticks="outside", ticklen=4,
        outlinecolor=INK, outlinewidth=1,
    )

    fig.update_layout(
        coloraxis=dict(
            colorscale="Jet", cmin=0, cmax=1,
            colorbar={
                **shared_colorbar,
                "x": LEFT_CENTER, "xanchor": "center",
                "tickvals": [0, .25, .5, .75, 1],
                "ticktext": ["0", ".25", ".50", ".75", "1"],
            },
        ),
        coloraxis2=dict(
            colorscale="Viridis", cmin=0, cmax=100,
            colorbar={
                **shared_colorbar,
                "x": RIGHT_CENTER, "xanchor": "center",
                "tickvals": [0, 25, 50, 75, 100],
                "ticktext": ["0", "25", "50", "75", "100"],
            },
        ),
        template="plotly_white",
        width=2300, height=1500,
        title=None,
        margin=dict(l=150, r=95, t=155, b=125),
        font={**FONT, "size": 17},
        paper_bgcolor="white",
        plot_bgcolor="#E9EDF2",
    )

    fig.show()

    return save_figure(
        fig, FIG_SUPP,
        "section1_mask_progression_r_and_paired_N_5pct",
        2300, 1500,
    )


SUPP_MASK_PROGRESSION = mask_progression_surface()

In [20]:
SLICE_TAUS = [0, 10, 25, 50, 75, 90]
LABEL_WINDOWS = [4, 14, 30]


def plot_window_slices(metric, ylabel, folder, stem):
    is_r = metric == "r"
    is_n = metric == "N_pct"
    fig = make_subplots(
        rows=2, cols=3, shared_xaxes=True, shared_yaxes=True,
        horizontal_spacing=.06, vertical_spacing=.16,
        subplot_titles=[f"SC τ={tau}%" for tau in SLICE_TAUS],
    )
    for heading in fig.layout.annotations:
        heading.font = dict(family="Arial", size=19, color=INK)

    for panel, tau in enumerate(SLICE_TAUS):
        row, col = panel // 3 + 1, panel % 3 + 1
        for code in ORDERED_MASKS:
            q = SENS_PRIMARY[
                SENS_PRIMARY["mask"].eq(code) & SENS_PRIMARY["tau"].eq(float(tau))
            ].sort_values("window_days")
            shown = q[metric].where(q[metric] > 0) if is_n else q[metric]
            fig.add_trace(
                go.Scatter(
                    x=q.window_days, y=shown, mode="lines",
                    name=f"{code} {GROUP_LOOKUP[code]['label']}",
                    legendgroup=code, showlegend=(panel == 0), connectgaps=False,
                    line=dict(color=MASK_COLORS[code], width=2.4),
                    customdata=np.column_stack([
                        q.N_pct, q.direct_pct, q.mean_window_support_pct,
                    ]),
                    hovertemplate=(
                        f"W=%{{x}} d<br>{ylabel}=%{{y:.3f}}"
                        "<br>Paired estimates=%{customdata[0]:.1f}%"
                        "<br>Direct dates=%{customdata[1]:.1f}%"
                        "<br>Mean direct support=%{customdata[2]:.1f}%"
                        "<extra>%{fullData.name}</extra>"
                    ),
                ), row=row, col=col,
            )

        for code in ("G1", "G8"):
            q = SENS_PRIMARY[
                SENS_PRIMARY["mask"].eq(code)
                & SENS_PRIMARY["tau"].eq(float(tau))
                & SENS_PRIMARY.window_days.isin(LABEL_WINDOWS)
            ].sort_values("window_days")
            for _, point in q.iterrows():
                value = point[metric]
                if not np.isfinite(value) or (is_n and value <= 0):
                    continue
                above = code == ("G8" if is_n else "G1")
                fig.add_annotation(
                    x=point.window_days, y=value,
                    text=f"{value:.1f}%" if is_n else f"{value:.2f}",
                    showarrow=True, arrowhead=0, arrowwidth=.75,
                    arrowcolor="#A9B9C9", ax=0, ay=-22 if above else 22,
                    font=dict(size=13, color=MASK_COLORS[code]),
                    bgcolor="white", bordercolor=MASK_COLORS[code],
                    borderwidth=.8, borderpad=2, row=row, col=col,
                )

        fig.update_xaxes(
            range=[1, 30], tickvals=[1, 4, 7, 14, 21, 30],
            gridcolor=GRID, showline=True, linecolor=INK, linewidth=1.4,
            ticks="outside", tickfont=dict(size=15), row=row, col=col,
        )
        y_style = dict(
            gridcolor=GRID, zeroline=False, showline=True, linecolor=INK,
            linewidth=1.4, ticks="outside", tickfont=dict(size=15),
            showticklabels=(col == 1),
        )
        if is_n:
            y_style.update(
                type="log", range=[-2, 2.15],
                tickvals=[.01, .1, 1, 10, 100],
                ticktext=["0.01", "0.1", "1", "10", "100"],
            )
        else:
            y_style.update(range=[-1.05, 1.05], tickvals=[-1, -.5, 0, .5, 1])
        fig.update_yaxes(row=row, col=col, **y_style)

    fig.add_annotation(
        x=-.065, y=.5, xref="paper", yref="paper",
        text="N% paired estimates (log scale)" if is_n else ylabel,
        textangle=-90, showarrow=False, font=dict(size=18, color=INK),
    )
    fig.add_annotation(
        x=.5, y=-.09, xref="paper", yref="paper",
        text="Centred common calendar window (days)",
        showarrow=False, font=dict(size=18, color=INK),
    )
    fig.update_layout(
        template=PLOT_TEMPLATE, width=1800, height=850,
        legend=dict(x=1.015, y=1, xanchor="left", yanchor="top", font=dict(size=15)),
        margin=dict(l=95, r=230, t=100, b=95),
        font=dict(family="Arial", color=INK, size=17),
        paper_bgcolor="white", plot_bgcolor="white",
    )
    fig.show()
    return save_figure(fig, folder, stem, 1800, 850)


MAIN_WINDOW_R_SLICES = plot_window_slices(
    "r", "Pearson r", FIG_MAIN, "section1_window_trajectories_r"
)
SUPP_WINDOW_N_SLICES = plot_window_slices(
    "N_pct", "N% paired estimates", FIG_SUPP, "section1_window_trajectories_N"
)


## 1.6 Response-ready conclusion for the temporal-window critique

The revised analysis no longer treats the centred 30-day mean as a default daily product. Configuration selection is performed on native-daily observations, and window length is evaluated separately as an application-dependent sensitivity parameter. Across the tested windows, longer aggregation increases the proportion of dates for which a paired estimate can be formed, because neighbouring valid observations contribute to overlapping centred windows. This improves continuity but does not create additional direct observations and reduces temporal specificity. We therefore report direct-date and paired-estimate coverage separately, retain the native-daily results as the primary reliability evidence, and use short and long windows only to show the trade-off between temporal support and event-scale interpretation.

**Novelty link:** the contribution is the explicit decision sequence and separation of observability from smoothing, which allows the same reliability questions to be applied to other hazards, regions, and validation data rather than prescribing a single aggregation period.


# 2. Composite score and configuration selection

## Analytical question

Can configuration choice be made transparent using only alignment and usable coverage, without duplicating error information through RMSE?

## Revision decision

The revised ranking score is

\[
S = \left(\frac{N}{100}\right)\max(r,0)^2.
\]

`N%` rewards temporal availability, while `r²` emphasizes explained co-variation and prevents negative alignment from receiving a positive rank. The score is a **ranking device**, not a reliability probability or universal acceptance criterion. `r` and `N%` remain visible in every selection table and figure.

This addresses **R1.2, R2.2, R3.4, R3.6, and R3.7**, and supports the framework argument under **R1.3, R2.1, and E1**.


In [72]:
DAILY_BY_REGION = []
for region in REGION_ORDER:
    daily = evaluate_region(region, windows=[1], sc_grid=SC_GRID)
    DAILY_BY_REGION.append(daily)
DAILY_ALL = pd.concat(DAILY_BY_REGION, ignore_index=True)
DAILY_ALL.to_csv(TABLE_SUPP / "all_subgrids_daily_mask_sc_results.csv.gz", index=False, compression="gzip")
print(f"{len(DAILY_ALL):,} daily five-sub-grid configurations calculated")

assert np.allclose(DAILY_ALL.N_pct, DAILY_ALL.direct_pct, equal_nan=True), (
    "At W=1, paired-estimate and direct-date coverage should match"
)


4,040 daily five-sub-grid configurations calculated


## 2.1 Main-paper evidence — regional alignment–coverage trade-off

**Reviewer IDs:** R1.2, R2.1, R2.2, R2.3, R3.3, R3.4, R3.6, R3.7, and E1.

**Placement:** Main paper. This is the clearest visual explanation of what the framework selects.

**How to read the result.** Each coloured trajectory follows one GHSL mask as the spatial-completeness threshold changes. Points farther right retain more dates; points higher up align more strongly with NGCP load. The star marks the configuration selected by `N × r²`. A defensible setting lies near the upper-right boundary, but the exact balance is region-specific.

**Main-paper interpretation.** The curves show that reliability is not maximized by correlation alone. Tight thresholds may retain too few dates, while permissive settings may increase coverage at the cost of weaker alignment. The selected configurations occupy a compromise frontier, turning the framework into an explicit and inspectable decision process.

**Transferability claim.** The same workflow can be applied across all five Visayas sub-grids, but the optimum is not assumed to be spatially universal.


In [76]:
# ============================================================
# Native-daily alignment–coverage trade-off
# W = 1; τ = 5, 10, ..., 95%
# ============================================================

TAUS_5 = np.arange(5, 100, 5)

DAILY_TRADEOFF = DAILY_ALL[
    DAILY_ALL.window_days.eq(1)
    & DAILY_ALL.tau.isin(TAUS_5)
].copy()

# Common axes preserve comparability while reducing empty space.
xmax = min(100, max(10, np.ceil(DAILY_TRADEOFF.N_pct.max() / 10) * 10))
ymin = max(-1, np.floor(DAILY_TRADEOFF.r.min() * 10) / 10 - .05)

fig = make_subplots(
    rows=1, cols=5, shared_xaxes=True, shared_yaxes=True,
    horizontal_spacing=.035, subplot_titles=REGION_ORDER,
)

for heading in fig.layout.annotations:
    heading.font = dict(family="Arial", size=20, color=INK)

for col, region in enumerate(REGION_ORDER, start=1):
    regional = DAILY_TRADEOFF[DAILY_TRADEOFF.region.eq(region)]

    # One trajectory per GHSL mask.
    for code in ORDERED_MASKS:
        q = regional[regional["mask"].eq(code)].sort_values("tau")

        fig.add_trace(
            go.Scatter(
                x=q.N_pct, y=q.r,
                mode="lines+markers",
                name=f"{code}: {GROUP_LOOKUP[code]['label']}",
                legendgroup=code,
                showlegend=(col == 1),
                line=dict(color=MASK_COLORS[code], width=1.6),
                marker=dict(
                    color=MASK_COLORS[code], size=8, opacity=.9,
                    line=dict(color=INK, width=.65),
                ),
                customdata=np.column_stack([q.tau, q.score, q.n_pairs]),
                hovertemplate=(
                    f"{region} · {code}<br>"
                    "SC threshold τ=%{customdata[0]:.0f}%<br>"
                    "Pearson r=%{y:.3f}<br>"
                    "Direct-date N=%{x:.1f}%<br>"
                    "Composite score=%{customdata[1]:.4f}<br>"
                    "Paired dates=%{customdata[2]:.0f}"
                    "<extra></extra>"
                ),
            ),
            row=1, col=col,
        )

    # Highest composite-score configuration in this region.
    best = regional.sort_values(
        ["score", "N_pct", "r", "tau", "mask"],
        ascending=[False, False, False, True, True],
    ).iloc[0]

    fig.add_trace(
        go.Scatter(
            x=[best.N_pct], y=[best.r],
            mode="markers+text",
            name="Selected configuration",
            legendgroup="selected",
            showlegend=(col == 1),
            text=[f"τ={best.tau:.0f}%"],
            textposition="top center",
            textfont=dict(family="Arial", size=13, color=INK),
            marker=dict(
                symbol="star", size=17,
                color=MASK_COLORS[best["mask"]],
                line=dict(color="black", width=1.3),
            ),
            customdata=[[
                best["mask"], best["mask_label"],
                best["score"], best["n_pairs"],
            ]],
            hovertemplate=(
                f"{region}<br>"
                "%{customdata[0]}: %{customdata[1]}<br>"
                f"SC threshold τ={best.tau:.0f}%<br>"
                "Pearson r=%{y:.3f}<br>"
                "Direct-date N=%{x:.1f}%<br>"
                "Composite score=%{customdata[2]:.4f}<br>"
                "Paired dates=%{customdata[3]:.0f}"
                "<extra>Selected configuration</extra>"
            ),
        ),
        row=1, col=col,
    )

    # Interpretive guides—not acceptance thresholds.
    fig.add_hline(
        y=.80, row=1, col=col,
        line_dash="dot", line_color="#A9B9C9", line_width=1,
    )
    fig.add_vline(
        x=50, row=1, col=col,
        line_dash="dot", line_color="#A9B9C9", line_width=1,
    )

    fig.update_xaxes(
        range=[0, xmax + 2],
        title_text="Direct-date N (%)",
        title_font=dict(family="Arial", size=17, color=INK),
        tickfont=dict(family="Arial", size=14, color=INK),
        gridcolor=GRID, showline=True, mirror=True,
        linecolor=INK, linewidth=1.3,
        ticks="outside", ticklen=5,
        row=1, col=col,
    )

    fig.update_yaxes(
        range=[ymin, 1.03],
        title_text="Pearson r" if col == 1 else None,
        title_font=dict(family="Arial", size=18, color=INK),
        tickfont=dict(family="Arial", size=14, color=INK),
        gridcolor=GRID, zeroline=True, zerolinecolor="#A9B9C9",
        showline=True, mirror=True,
        linecolor=INK, linewidth=1.3,
        ticks="outside", ticklen=5,
        row=1, col=col,
    )

fig.update_layout(
    template="plotly_white",
    width=2200, height=590,
    title=None,
    legend=dict(
        x=1.01, xanchor="left",
        y=1, yanchor="top",
        font=dict(family="Arial", size=13, color=INK),
        tracegroupgap=3,
    ),
    margin=dict(l=90, r=260, t=60, b=85),
    font=dict(family="Arial", color=INK, size=15),
    paper_bgcolor="white",
    plot_bgcolor="white",
)

fig.show()

SUPP_TRADEOFF = save_figure(
    fig, FIG_SUPP,
    "section2_r_N_tradeoff_by_subgrid",
    2200, 590,
)


# ============================================================
# Top five daily configurations per sub-grid
# Uses the exact configurations displayed above
# ============================================================

TOP5 = (
    DAILY_TRADEOFF
    .sort_values(
        ["region", "score", "N_pct", "r", "tau", "mask"],
        ascending=[True, False, False, False, True, True],
    )
    .groupby("region", group_keys=False)
    .head(5)
    .copy()
)

TOP5["rank"] = TOP5.groupby("region").cumcount() + 1
TOP5["region"] = pd.Categorical(
    TOP5["region"], categories=REGION_ORDER, ordered=True,
)

TOP5_MAIN = (
    TOP5[
        [
            "region", "rank", "mask", "mask_label", "tau",
            "score", "r", "N_pct", "n_pairs",
        ]
    ]
    .sort_values(["region", "rank"])
    .reset_index(drop=True)
)

display(TOP5_MAIN.round({
    "tau": 0, "score": 4, "r": 3, "N_pct": 1,
}))

TOP5_MAIN.to_csv(
    TABLE_MAIN / "section2_top5_configurations_by_subgrid.csv",
    index=False,
)

try:
    TOP5_MAIN.to_latex(
        TABLE_MAIN / "section2_top5_configurations_by_subgrid.tex",
        index=False,
        float_format="%.3f",
    )
except Exception as exc:
    print("LaTeX export skipped:", exc)

,region,rank,mask,mask_label,tau,score,r,N_pct,n_pairs
0,Samar-Leyte,1,G1,Urban Core,25.0,0.1600,0.602,44.2,1936
1,Samar-Leyte,2,G1,Urban Core,30.0,0.1575,0.615,41.6,1823
2,Samar-Leyte,3,G1,Urban Core,20.0,0.1572,0.579,46.9,2057
3,Samar-Leyte,4,G1,Urban Core,15.0,0.1560,0.559,50.0,2191
4,Samar-Leyte,5,G1,Urban Core,40.0,0.1546,0.650,36.6,1605
5,Bohol,1,G1,Urban Core,40.0,0.1267,0.660,29.1,1275
6,Bohol,2,G1,Urban Core,35.0,0.1256,0.649,29.8,1307
7,Bohol,3,G1,Urban Core,45.0,0.1252,0.664,28.4,1244
8,Bohol,4,G1,Urban Core,50.0,0.1221,0.665,27.6,1210
9,Bohol,5,G1,Urban Core,30.0,0.1207,0.628,30.6,1342


## 2.2 Regional component and score surfaces

**Reviewer IDs:** R1.2, R2.2, R2.3, R3.3, R3.4, R3.6, R3.7.

**Placement:** Composite-score surface in the main paper if space permits; component `r` and `N%` surfaces in the supplementary materials.

**How to read the result.** The common five-percent threshold grid permits direct comparison across sub-grids. The component surfaces show why a setting scores well; the score surface summarizes where the balance is strongest. Region-level patterns should be compared structurally, not by interpreting small colour differences as inferentially significant.

**Main-paper interpretation.** Urban-focused masks recur among high-scoring settings, but selected spatial-completeness thresholds differ between sub-grids. This is evidence for a transferable workflow with locally calibrated parameters—not for one fixed Visayas-wide configuration.


In [75]:
# ============================================================
# Compact five-sub-grid native-daily heatmaps
# Same styling and colour references as annotated heatmaps
# ============================================================

DISPLAY_TAUS_5 = np.arange(5, 100, 5)


def regional_daily_heatmaps(metric, destination):
    daily = DAILY_ALL[
        DAILY_ALL.window_days.eq(1)
        & DAILY_ALL.tau.isin(DISPLAY_TAUS_5)
    ].copy()

    if metric == "r":
        colorscale, cmin, cmax = "Blues_r", 0, .60
        metric_title, value_label = "Pearson r", "Pearson r"
        ticks = [0, .20, .40, .60]
        ticktext = ["0", ".20", ".40","≥.60"]

    elif metric == "N_pct":
        colorscale, cmin, cmax = "Greens_r", 0, 50
        metric_title, value_label = "Direct-date N%", "Direct N"
        ticks = [0, 10, 20, 30, 40, 50]
        ticktext = ["0", "10", "20", "30", "40", "≥50"]

    elif metric == "score":
        colorscale, cmin = "jet", 0
        cmax = max(.01, float(daily.score.max()))
        metric_title, value_label = "Composite score", "Score"
        ticks = np.linspace(cmin, cmax, 5)
        ticktext = [f"{value:.2f}" for value in ticks]

    else:
        raise ValueError("metric must be 'r', 'N_pct', or 'score'")

    fig = make_subplots(
        rows=1, cols=5, shared_yaxes=True,
        horizontal_spacing=.03,
        subplot_titles=REGION_ORDER,
    )

    y_labels = [
        f"{code}: {GROUP_LOOKUP[code]['label']}"
        for code in ORDERED_MASKS
    ]

    for col, region in enumerate(REGION_ORDER, start=1):
        subset = daily[daily.region.eq(region)]

        matrix = (
            subset
            .pivot(index="mask", columns="tau", values=metric)
            .reindex(index=ORDERED_MASKS, columns=DISPLAY_TAUS_5)
        )

        raw = matrix.to_numpy(dtype=float)

        fig.add_trace(
            go.Heatmap(
                x=matrix.columns,
                y=y_labels,
                z=raw,
                customdata=raw,
                coloraxis="coloraxis",
                xgap=2,
                ygap=2,
                hoverongaps=False,
                hovertemplate=(
                    f"{region}<br>"
                    "Mask=%{y}<br>"
                    "SC threshold τ=%{x:.0f}%<br>"
                    f"{value_label}=%{{customdata:.3f}}"
                    "<extra></extra>"
                ),
            ),
            row=1, col=col,
        )

        fig.update_xaxes(
            range=[2.5, 97.5],
            title_text="SC threshold, τ (%)",
            title_font=dict(family="Arial", size=17, color=INK),
            tickvals=[20, 40, 60, 80],
            ticktext=["20", "40", "60", "80"],
            tickfont=dict(family="Arial", size=15, color=INK),
            showline=True, mirror=True,
            linecolor=INK, linewidth=1.4,
            ticks="outside", ticklen=5,
            row=1, col=col,
        )

        fig.update_yaxes(
            title_text="GHSL thematic mask" if col == 1 else None,
            title_font=dict(family="Arial", size=18, color=INK),
            tickfont=dict(family="Arial", size=14, color=INK),
            showticklabels=(col == 1),
            showline=True, mirror=True,
            linecolor=INK, linewidth=1.4,
            ticks="outside", ticklen=5,
            row=1, col=col,
        )

    for heading in fig.layout.annotations:
        heading.font = dict(family="Arial", size=20, color=INK)

    colorbar = dict(
        title=dict(
            text=metric_title,
            side="right",
            font=dict(family="Arial", size=17, color=INK),
        ),
        x=1.01, y=.5,
        len=.82, thickness=22,
        tickvals=ticks,
        ticktext=ticktext,
        tickfont=dict(family="Arial", size=14, color=INK),
        ticks="outside", ticklen=4,
        outlinecolor=INK, outlinewidth=1,
    )

    fig.update_layout(
        template="plotly_white",
        width=2300, height=520,
        title=None,
        coloraxis=dict(
            colorscale=colorscale,
            cmin=cmin, cmax=cmax,
            colorbar=colorbar,
        ),
        margin=dict(l=235, r=120, t=60, b=90),
        font=dict(family="Arial", color=INK, size=16),
        paper_bgcolor="white",
        plot_bgcolor="white",
    )

    fig.show()

    folder = FIG_MAIN if destination == "main" else FIG_SUPP
    return save_figure(
        fig, folder,
        f"section2_daily_{metric}_by_subgrid",
        2300, 520,
    )


SUPP_DAILY_R = regional_daily_heatmaps("r", "supplement")
SUPP_DAILY_N = regional_daily_heatmaps("N_pct", "supplement")
MAIN_DAILY_SCORE = regional_daily_heatmaps("score", "main")

## 2.3 Configuration tables

**Reviewer IDs:** R1.2, R2.2, R2.3, R3.3, R3.4.

**Recommended placement:**

- **Main paper:** one best configuration per sub-grid, reporting mask, threshold, `r`, `N%`, paired days, and score.
- **Supplementary materials:** the five highest-ranked configurations per sub-grid, because adjacent thresholds can produce near-equivalent scores.

The top-five table is a robustness view, not five independent optima. Similar scores across neighbouring thresholds indicate a stable selection neighbourhood rather than precision to a single percentage point.


In [39]:
# ============================================================
# Main Table — Top five daily configurations per sub-grid
# Restricted to τ = 5, 10, ..., 95%
# ============================================================

DISPLAY_TAUS_5 = np.arange(5, 100, 5, dtype=float)

REGIONAL_DAILY_5PCT = (
    DAILY_ALL[
        DAILY_ALL.window_days.eq(1)
        & DAILY_ALL.tau.isin(DISPLAY_TAUS_5)
    ]
    .copy()
)

# Preserve the intended regional order.
REGIONAL_DAILY_5PCT["region"] = pd.Categorical(
    REGIONAL_DAILY_5PCT["region"],
    categories=REGION_ORDER,
    ordered=True,
)

# Rank using the revised daily composite score.
REGIONAL_DAILY_5PCT = (
    REGIONAL_DAILY_5PCT
    .sort_values(
        [
            "region",
            "score",
            "N_pct",
            "r",
            "tau",
            "mask",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            True,
            True,
        ],
    )
)

TOP5_CONFIGS_BY_REGION = (
    REGIONAL_DAILY_5PCT
    .groupby(
        "region",
        observed=True,
        group_keys=False,
    )
    .head(5)
    .copy()
)

TOP5_CONFIGS_BY_REGION["rank"] = (
    TOP5_CONFIGS_BY_REGION
    .groupby(
        "region",
        observed=True,
    )
    .cumcount()
    + 1
)

TOP5_CONFIGS_BY_REGION = (
    TOP5_CONFIGS_BY_REGION[
        [
            "region",
            "rank",
            "mask",
            "mask_label",
            "tau",
            "r",
            "N_pct",
            "n_pairs",
            "score",
        ]
    ]
    .sort_values(
        ["region", "rank"]
    )
    .reset_index(drop=True)
)

TOP5_CONFIGS_DISPLAY = (
    TOP5_CONFIGS_BY_REGION
    .rename(
        columns={
            "region": "Sub-grid",
            "rank": "Rank",
            "mask": "GHSL mask",
            "mask_label": "Mask description",
            "tau": "SC τ (%)",
            "r": "Pearson r",
            "N_pct": "N (%)",
            "n_pairs": "Paired days",
            "score": "Composite score",
        }
    )
    .copy()
)

TOP5_CONFIGS_DISPLAY["SC τ (%)"] = (
    TOP5_CONFIGS_DISPLAY["SC τ (%)"]
    .round(0)
    .astype(int)
)

TOP5_CONFIGS_DISPLAY["Pearson r"] = (
    TOP5_CONFIGS_DISPLAY["Pearson r"]
    .round(3)
)

TOP5_CONFIGS_DISPLAY["N (%)"] = (
    TOP5_CONFIGS_DISPLAY["N (%)"]
    .round(1)
)

TOP5_CONFIGS_DISPLAY["Composite score"] = (
    TOP5_CONFIGS_DISPLAY["Composite score"]
    .round(4)
)

display(
    Markdown(
        "## Top five native-daily configurations "
        "per sub-grid (5% τ grid)"
    )
)

display(TOP5_CONFIGS_DISPLAY)

TOP5_CONFIGS_DISPLAY.to_csv(
    TABLE_MAIN
    / "section2_top5_daily_configurations_by_subgrid_5pct_tau.csv",
    index=False,
)

TOP5_CONFIGS_BY_REGION.to_csv(
    TABLE_SUPP
    / "section2_top5_daily_configurations_by_subgrid_5pct_tau_raw.csv",
    index=False,
)

## Top five native-daily configurations per sub-grid (5% τ grid)

,Sub-grid,Rank,GHSL mask,Mask description,SC τ (%),Pearson r,N (%),Paired days,Composite score
0,Samar-Leyte,1,G1,Urban Core,25,0.602,44.2,1936,0.1600
1,Samar-Leyte,2,G1,Urban Core,30,0.615,41.6,1823,0.1575
2,Samar-Leyte,3,G1,Urban Core,20,0.579,46.9,2057,0.1572
3,Samar-Leyte,4,G1,Urban Core,15,0.559,50.0,2191,0.1560
4,Samar-Leyte,5,G1,Urban Core,40,0.650,36.6,1605,0.1546
5,Bohol,1,G1,Urban Core,40,0.660,29.1,1275,0.1267
6,Bohol,2,G1,Urban Core,35,0.649,29.8,1307,0.1256
7,Bohol,3,G1,Urban Core,45,0.664,28.4,1244,0.1252
8,Bohol,4,G1,Urban Core,50,0.665,27.6,1210,0.1221
9,Bohol,5,G1,Urban Core,30,0.628,30.6,1342,0.1207


## 2.4 Supplementary evidence — score-form sensitivity

**Reviewer IDs:** R1.2, R2.2, R3.4.

**Placement:** Supplementary materials.

**How to read the result.** Each point is the same daily configuration ranked under two plausible score forms. Points close to the diagonal receive similar relative scores. The labelled winners show whether using `N × r` or `N × r²` changes the selected mask or threshold; Spearman's `ρ` summarizes whole-ranking agreement.

**Supplementary interpretation.** High rank agreement in most sub-grids shows that the broad configuration ordering is not created solely by squaring `r`. Where the winning threshold changes, `r²` deliberately places more emphasis on alignment. This is a meaningful sensitivity result because it makes the weighting consequence explicit; it is not evidence that one formula is universally correct.

**Cautious claim.** Formula sensitivity supports transparency and robustness of the selection logic, but does not replace an application-specific statement of whether coverage or alignment is more consequential.


In [78]:
# ============================================================
# Supplementary score-form sensitivity
# Compare N×r against N×r²
# ============================================================

TAUS_5 = np.arange(5, 100, 5)

SCORE_SENS = DAILY_ALL[
    DAILY_ALL.window_days.eq(1)
    & DAILY_ALL.tau.isin(TAUS_5)
].copy()

SCORE_SENS["r_pos"] = SCORE_SENS.r.clip(lower=0)
SCORE_SENS["score_Nr"] = (SCORE_SENS.N_pct / 100) * SCORE_SENS.r_pos
SCORE_SENS["score_Nr2"] = (SCORE_SENS.N_pct / 100) * SCORE_SENS.r_pos.pow(2)

for score in ["score_Nr", "score_Nr2"]:
    SCORE_SENS[f"{score}_norm"] = SCORE_SENS.groupby("region")[score].transform(
        lambda s: s / s.max() if s.max() > 0 else np.zeros(len(s))
    )
    SCORE_SENS[f"{score}_rank"] = SCORE_SENS.groupby("region")[score].rank(
        method="min", ascending=False
    )

fig = make_subplots(
    rows=1, cols=5, shared_xaxes=True, shared_yaxes=True,
    horizontal_spacing=.035, subplot_titles=REGION_ORDER,
)

for heading in fig.layout.annotations:
    heading.font = dict(family="Arial", size=20, color=INK)

selection_rows = []

for col, region in enumerate(REGION_ORDER, start=1):
    q = SCORE_SENS[SCORE_SENS.region.eq(region)].copy()
    rho = q[["score_Nr_rank", "score_Nr2_rank"]].corr(
        method="spearman"
    ).iloc[0, 1]

    winners = {
        "N × r": q.sort_values(
            ["score_Nr", "N_pct", "r", "tau", "mask"],
            ascending=[False, False, False, True, True],
        ).iloc[0],
        "N × r²": q.sort_values(
            ["score_Nr2", "N_pct", "r", "tau", "mask"],
            ascending=[False, False, False, True, True],
        ).iloc[0],
    }

    # Background configuration cloud.
    for code in ORDERED_MASKS:
        m = q[q["mask"].eq(code)]

        fig.add_trace(
            go.Scatter(
                x=m.score_Nr_norm, y=m.score_Nr2_norm,
                mode="markers",
                name=f"{code}: {GROUP_LOOKUP[code]['label']}",
                legendgroup=code, showlegend=(col == 1),
                marker=dict(
                    color=MASK_COLORS[code], size=7, opacity=.58,
                    line=dict(color=INK, width=.4),
                ),
                customdata=np.column_stack([
                    m.tau, m.r, m.N_pct,
                    m.score_Nr, m.score_Nr2,
                    m.score_Nr_rank, m.score_Nr2_rank,
                ]),
                hovertemplate=(
                    f"{region} · {code}<br>"
                    "τ=%{customdata[0]:.0f}%"
                    "<br>r=%{customdata[1]:.3f}"
                    " · N=%{customdata[2]:.1f}%"
                    "<br>N×r=%{customdata[3]:.4f}"
                    " · rank %{customdata[5]:.0f}"
                    "<br>N×r²=%{customdata[4]:.4f}"
                    " · rank %{customdata[6]:.0f}"
                    "<extra></extra>"
                ),
            ),
            row=1, col=col,
        )

    winner_styles = {
        "N × r": dict(symbol="star", size=19, color="#D65A4A"),
        "N × r²": dict(symbol="diamond-open", size=21, color=INK),
    }

    for rule, winner in winners.items():
        fig.add_trace(
            go.Scatter(
                x=[winner.score_Nr_norm],
                y=[winner.score_Nr2_norm],
                mode="markers",
                name=f"Selected by {rule}",
                legendgroup=f"selected_{rule}",
                showlegend=(col == 1),
                marker={
                    **winner_styles[rule],
                    "line": dict(color="black", width=1.5),
                },
                hovertemplate=(
                    f"{region}<br>{rule} winner"
                    f"<br>{winner['mask']}: {winner['mask_label']}"
                    f"<br>τ={winner.tau:.0f}%"
                    f"<br>r={winner.r:.3f}"
                    f" · N={winner.N_pct:.1f}%"
                    "<extra></extra>"
                ),
            ),
            row=1, col=col,
        )

        score_col = "score_Nr" if rule == "N × r" else "score_Nr2"
        selection_rows.append({
            "region": region,
            "rule": rule,
            "mask": winner["mask"],
            "mask_label": winner["mask_label"],
            "tau": winner["tau"],
            "r": winner["r"],
            "N_pct": winner["N_pct"],
            "score": winner[score_col],
            "rank_rho": rho,
        })

    # One-to-one reference.
    fig.add_shape(
        type="line", x0=0, y0=0, x1=1, y1=1,
        line=dict(color="#A9B9C9", width=1.2, dash="dot"),
        row=1, col=col,
    )

    axis = "" if col == 1 else str(col)
    winner_Nr, winner_Nr2 = winners["N × r"], winners["N × r²"]

    # Scientifically useful annotation: actual winners.
    fig.add_annotation(
        x=.035, y=.965,
        xref=f"x{axis} domain", yref=f"y{axis} domain",
        text=(
            f"Rank ρ = {rho:.2f}"
            f"<br>N×r: {winner_Nr['mask']} · τ={winner_Nr.tau:.0f}%"
            f"<br>N×r²: {winner_Nr2['mask']} · τ={winner_Nr2.tau:.0f}%"
        ),
        showarrow=False, xanchor="left", yanchor="top", align="left",
        bgcolor="rgba(255,255,255,.90)",
        bordercolor="#A9B9C9", borderwidth=.8, borderpad=5,
        font=dict(family="Arial", size=13, color=INK),
    )

    fig.update_xaxes(
        range=[0, 1.03], tickvals=[0, .25, .5, .75, 1],
        title_text="Relative N × r score",
        title_font=dict(family="Arial", size=16, color=INK),
        tickfont=dict(family="Arial", size=13, color=INK),
        gridcolor=GRID, showline=True, mirror=True,
        linecolor=INK, linewidth=1.3, ticks="outside",
        row=1, col=col,
    )

    fig.update_yaxes(
        range=[0, 1.03], tickvals=[0, .25, .5, .75, 1],
        title_text="Relative N × r² score" if col == 1 else None,
        title_font=dict(family="Arial", size=17, color=INK),
        tickfont=dict(family="Arial", size=13, color=INK),
        gridcolor=GRID, showline=True, mirror=True,
        linecolor=INK, linewidth=1.3, ticks="outside",
        row=1, col=col,
    )

fig.update_layout(
    template="plotly_white",
    width=2200, height=590,
    title=None,
    legend=dict(
        x=1.01, xanchor="left", y=1, yanchor="top",
        font=dict(family="Arial", size=13, color=INK),
    ),
    margin=dict(l=85, r=275, t=60, b=90),
    font=dict(family="Arial", color=INK, size=15),
    paper_bgcolor="white", plot_bgcolor="white",
)

fig.show()

SUPP_SCORE_COMPARISON = save_figure(
    fig, FIG_SUPP,
    "section2_composite_score_Nr_vs_Nr2",
    2200, 590,
)

SCORE_SELECTIONS = pd.DataFrame(selection_rows)

display(SCORE_SELECTIONS.round({
    "tau": 0, "r": 3, "N_pct": 1,
    "score": 4, "rank_rho": 3,
}))

SCORE_SELECTIONS.to_csv(
    TABLE_SUPP / "section2_composite_score_Nr_vs_Nr2_selections.csv",
    index=False,
)

,region,rule,mask,mask_label,tau,r,N_pct,score,rank_rho
0,Samar-Leyte,N × r,G2,Dense Urban,5.0,0.466,63.2,0.2945,0.945
1,Samar-Leyte,N × r²,G1,Urban Core,25.0,0.602,44.2,0.1600,0.945
2,Bohol,N × r,G4,Urban Fringe,5.0,0.440,53.1,0.2339,0.884
3,Bohol,N × r²,G1,Urban Core,40.0,0.660,29.1,0.1267,0.884
4,Cebu,N × r,G1,Urban Core,25.0,0.384,33.5,0.1284,0.368
5,Cebu,N × r²,G1,Urban Core,50.0,0.435,27.7,0.0525,0.368
6,Negros,N × r,G4,Urban Fringe,5.0,0.337,57.7,0.1944,0.908
7,Negros,N × r²,G1,Urban Core,15.0,0.424,41.8,0.0753,0.908
8,Panay,N × r,G4,Urban Fringe,5.0,0.406,56.9,0.2313,0.889
9,Panay,N × r²,G4,Urban Fringe,5.0,0.406,56.9,0.0940,0.889


## 2.5 Diagnostic archive — annotated regional surfaces

**Publication role:** retain as a detailed audit trail. The compact regional panels and trade-off figure communicate the result more efficiently, so the fifteen individual annotated heatmaps should not all be placed in the paper.

These plots are useful when checking a specific sub-grid, resolving close rankings, or responding to a request for exact cell values.


In [40]:
# ============================================================
# Regional native-daily heatmaps
# Pearson r, direct-date N%, and region-scaled composite score
# ============================================================

DISPLAY_TAUS_5 = np.arange(5, 100, 5, dtype=float)


def regional_mask_tau_matrix(
    frame,
    region,
    metric,
    display_taus=DISPLAY_TAUS_5,
):
    subset = frame[
        frame.region.eq(region)
        & frame.window_days.eq(1)
    ]

    return (
        subset
        .pivot(
            index="mask",
            columns="tau",
            values=metric,
        )
        .reindex(
            index=ORDERED_MASKS,
            columns=display_taus,
        )
    )


def plot_regional_daily_heatmap(
    frame,
    region,
    metric,
):
    matrix = regional_mask_tau_matrix(
        frame=frame,
        region=region,
        metric=metric,
    )

    z = matrix.to_numpy(dtype=float)

    y_labels = [
        f"{code}: {GROUP_LOOKUP[code]['label']}"
        for code in matrix.index
    ]

    # --------------------------------------------------------
    # Pearson r
    # --------------------------------------------------------

    if metric == "r":
        colorscale = "Blues_r"
        zmin, zmax = 0, 0.80

        # Same visual reference as Notebook 3.
        highlight = z >= 0.80
        text_format = "{:.2f}"

        colorbar = dict(
            title=dict(
                text="Pearson r",
                side="right",
            ),
            len=0.85,
            tickvals=[
                0,
                0.20,
                0.40,
                0.60,
                0.80,
            ],
            ticktext=[
                "0",
                "0.20",
                "0.40",
                "0.60",
                "≥0.80",
            ],
        )

        metric_name = "Pearson r"
        file_tag = "pearson_r"

    # --------------------------------------------------------
    # Direct-date N%
    # --------------------------------------------------------

    elif metric == "N_pct":
        colorscale = "Greens_r"
        zmin, zmax = 0, 50

        # Same visual reference as Notebook 3.
        highlight = z >= 50
        text_format = "{:.0f}"

        colorbar = dict(
            title=dict(
                text="N (%)",
                side="right",
            ),
            len=0.85,
            tickvals=[
                0,
                10,
                20,
                30,
                40,
                50,
            ],
            ticktext=[
                "0",
                "10",
                "20",
                "30",
                "40",
                "≥50",
            ],
        )

        metric_name = "Direct-date N%"
        file_tag = "direct_N"

    # --------------------------------------------------------
    # Composite score
    # --------------------------------------------------------

    elif metric == "score":
        colorscale = "Pinkyl_r"
        zmin = 0

        finite_scores = z[np.isfinite(z)]

        # Scale independently to the maximum of this region.
        zmax = (
            max(
                0.01,
                float(np.nanmax(finite_scores)),
            )
            if finite_scores.size
            else 0.01
        )

        # Bold the top-performing 10% within this region.
        regional_top_decile = (
            np.nanpercentile(
                finite_scores,
                90,
            )
            if finite_scores.size
            else np.nan
        )

        highlight = z >= regional_top_decile
        text_format = "{:.2f}"

        score_ticks = np.linspace(
            zmin,
            zmax,
            5,
        )

        colorbar = dict(
            title=dict(
                text=(
                    "Composite score"
                    "<br><sup>region-scaled</sup>"
                ),
                side="right",
            ),
            len=0.85,
            tickvals=score_ticks,
            ticktext=[
                f"{value:.3f}"
                for value in score_ticks
            ],
        )

        metric_name = "Composite score"
        file_tag = "composite_score"

    else:
        raise ValueError(
            "metric must be 'r', 'N_pct', or 'score'"
        )

    # --------------------------------------------------------
    # Heatmap
    # --------------------------------------------------------

    fig = go.Figure()

    fig.add_trace(
        go.Heatmap(
            z=z,
            x=matrix.columns,
            y=y_labels,
            colorscale=colorscale,
            zmin=zmin,
            zmax=zmax,
            xgap=2,
            ygap=2,
            colorbar=colorbar,
            showscale=True,
            hovertemplate=(
                f"{region}<br>"
                "Mask %{y}<br>"
                "Spatial completeness τ=%{x:.0f}%<br>"
                f"{metric_name}=%{{z:.3f}}"
                "<extra></extra>"
            ),
        )
    )

    # Annotate every cell.
    for row in range(z.shape[0]):
        for col in range(z.shape[1]):
            value = z[row, col]

            if not np.isfinite(value):
                continue

            value_text = text_format.format(value)

            if highlight[row, col]:
                value_text = f"<b>{value_text}</b>"

            fig.add_annotation(
                x=matrix.columns[col],
                y=y_labels[row],
                text=value_text,
                showarrow=False,
                font=dict(
                    family="Arial",
                    size=14,
                    color="black",
                ),
            )

    fig.update_layout(
        template="plotly_white",
        width=1500,
        height=400,
        title=None,
        margin=dict(
            l=175,
            r=45,
            t=20,
            b=80,
        ),
        xaxis=dict(
            title=(
                "Spatial completeness threshold, "
                "τ (%)"
            ),
            tickvals=list(
                range(0, 101, 20)
            ),
            tickfont=dict(size=14),
        ),
        yaxis=dict(
            tickfont=dict(size=14),
        ),
        font=dict(
            family="Arial",
            color=INK,
            size=16,
        ),
        paper_bgcolor="rgba(0,0,0,0)",
    )

    safe_region = (
        region.lower()
        .replace("–", "_")
        .replace("-", "_")
        .replace(" ", "_")
    )

    stem = (
        f"section2_{safe_region}_"
        f"native_daily_{file_tag}"
    )

    return fig, stem


# ============================================================
# Generate and export all regional panels
# ============================================================

ALL_REGIONAL_HEATMAPS = {}

for region in REGION_ORDER:
    display(
        Markdown(
            f"## {region}: native-daily reliability surfaces"
        )
    )

    fig_r, stem_r = plot_regional_daily_heatmap(
        frame=DAILY_ALL,
        region=region,
        metric="r",
    )

    fig_N, stem_N = plot_regional_daily_heatmap(
        frame=DAILY_ALL,
        region=region,
        metric="N_pct",
    )

    fig_score, stem_score = plot_regional_daily_heatmap(
        frame=DAILY_ALL,
        region=region,
        metric="score",
    )

    fig_r.show()
    fig_N.show()
    fig_score.show()

    saved_r = save_figure(
        fig_r,
        FIG_SUPP,
        stem_r,
        1500,
        400,
    )

    saved_N = save_figure(
        fig_N,
        FIG_SUPP,
        stem_N,
        1500,
        400,
    )

    saved_score = save_figure(
        fig_score,
        FIG_SUPP,
        stem_score,
        1500,
        400,
    )

    ALL_REGIONAL_HEATMAPS[region] = {
        "r": saved_r,
        "N_pct": saved_N,
        "score": saved_score,
    }

## Samar-Leyte: native-daily reliability surfaces

## Bohol: native-daily reliability surfaces

## Cebu: native-daily reliability surfaces

## Negros: native-daily reliability surfaces

## Panay: native-daily reliability surfaces

## 2.6 Response-ready conclusion for the composite-score critique

The original weighted composite was replaced with a two-component score, `S=(N/100)max(r,0)²`, because RMSE added a second expression of alignment error without an equally clear decision role. The revised analysis reports `r` and `N%` separately, uses the score only to rank mask–threshold configurations, and tests the consequence of the squared-correlation term against `N × r`. The regional trade-off curves show that the selected settings balance alignment and retained temporal coverage rather than maximizing either quantity alone. The same decision sequence transfers across the five sub-grids, while the selected parameters remain locally calibrated.

**Novelty link:** the framework formalizes how observability, spatial qualification, validation alignment, trade-off inspection, configuration selection, and sensitivity testing should be connected. Its novelty lies in that auditable sequence under severe cloud constraints, not in claiming novelty for Pearson correlation or coverage individually.


# 3. Publication and revision action list

## Evidence now available

| Manuscript action | Evidence to use | Status |
|---|---|---|
| Replace the unqualified 30-day rationale | Native-daily surface + window-effect summary + event comparison | Ready |
| Explain why long windows appear more complete | Direct-date versus paired-estimate distinction | Ready |
| Replace the original weighted score | `N × r²`, component reporting, and formula sensitivity | Ready |
| Demonstrate the framework across sub-grids | Trade-off curves, compact surfaces, and regional configuration table | Ready |
| Explain novelty beyond metric compilation | Ordered reliability decision sequence | Ready for manuscript wording |

## Remaining work before resubmission

1. **R3.8 — statistical uncertainty:** add time-series-aware uncertainty for selected correlations, preferably a block bootstrap or another dependence-preserving interval. Do not interpret small differences in `r` as significant until this is done.
2. **R1.3 / R2.1 / E1 — novelty comparison:** add a concise table or paragraph distinguishing the preliminary version from the journal revision, including the native-daily baseline, window sensitivity, revised score, regional transferability analysis, and decision framework.
3. **R3.1 / R3.2 — front matter:** rewrite the abstract and end of the introduction so the problem, framework, validation design, findings, and numbered contributions are explicit.
4. **R1.4 / R1.5 / R1.6 / R1.7 — presentation corrections:** complete the separate figure-contrast, temporal-coverage-definition, colour-scale, and line-readability edits in the manuscript.
5. **R2.3–R2.5 / R3.9 — scope and limitations:** restrict transferability claims to the tested Visayas sub-grids; distinguish signal representation from outage validation; state the retrospective nature of centred windows; and identify future event and sensor validation.
6. **R3.5 — GHSL temporal limitation:** state that the thematic mask is static for the analysis period and discuss possible land-cover or settlement change.
7. **Regional QA:** verify every sub-grid's mask files and NGCP sheet mapping, and investigate any exact duplicate winning configuration or metric row before final reporting.
8. **Table selection:** use one best setting per sub-grid in the main paper; place the top-five neighbourhood in the supplementary materials.
9. **Final editorial pass:** assign final figure/table numbers, update cross-references, and ensure captions distinguish direct observations from window-supported paired estimates.

## Final claim boundary

The analyses support a **reliability-qualification framework** for diagnosing when daily Black Marble NTL can be compared with an external load signal under severe cloud constraints. They do not establish a universal threshold, global transferability, causal disaster effects, or a real-time outage product.
